# 01 — EDA & Experiment Validation

**Goal of this notebook:** prove the experiment is *trustworthy* before we measure anything.

A p-value is only meaningful if the underlying experiment was sound. So before touching the
outcome columns, we answer three questions, in order:

1. **Is the data clean?** (types, missing values, impossible values)
2. **Did the randomizer deliver the intended split?** (Sample Ratio Mismatch check)
3. **Did randomization actually work?** (covariate balance: the three arms should look like
   three random samples of the *same* population)

Only if all three pass do we earn the right to compare outcomes in notebook 03.

**Dataset:** the Hillstrom email experiment — 64,000 real customers randomly assigned to
*Mens E-Mail*, *Womens E-Mail*, or *No E-Mail* (control). See `../data/README.md`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Shared statistical helpers (z-tests, SRM, balance checks) live in utils.py
# so the notebooks, the design module and the auditor all use ONE implementation.
from utils import srm_check, balance_check

sns.set_theme(style="whitegrid", context="notebook")

df = pd.read_csv("../data/raw/hillstrom.csv")
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## 1. Data quality

Boring on purpose. We check types, missing values, and value ranges. Any surprise here
(negative spend, recency of 40 months on a 12-month file...) would mean a data pipeline
problem to fix before any statistics.

In [ ]:
# Missing values: a randomized experiment export should have none.
missing = df.isna().sum()
print("Missing values per column:")
print(missing.to_string())
assert missing.sum() == 0, "Unexpected missing data - investigate before analysis"

# Range sanity checks on the numeric columns.
assert df["recency"].between(1, 12).all(), "recency outside the 12-month window"
assert (df["spend"] >= 0).all(), "negative spend should be impossible"
assert df["visit"].isin([0, 1]).all() and df["conversion"].isin([0, 1]).all()

# Logical funnel check: you cannot convert without visiting,
# and you cannot spend without converting.
assert (df.loc[df["conversion"] == 1, "visit"] == 1).all()
assert (df.loc[df["spend"] > 0, "conversion"] == 1).all()

print("\nAll integrity checks passed.")

In [ ]:
# The outcome funnel at a glance: visit -> conversion -> spend.
# These are POOLED numbers (all arms mixed) - purely descriptive at this stage.
print(f"visit rate      : {df['visit'].mean():.2%}")
print(f"conversion rate : {df['conversion'].mean():.3%}")
print(f"mean spend      : ${df['spend'].mean():.2f} per customer")
print(f"mean spend (buyers only): ${df.loc[df['spend']>0,'spend'].mean():.2f}")
print(f"share of customers with any spend: {(df['spend']>0).mean():.2%}")

Note how extreme the funnel is: ~15% visit, under 1% convert. This is *normal* for retail
email and it is exactly why sample size matters — a 0.9% base rate means the "signal" we
are hunting is tiny relative to the noise.

## 2. Sample Ratio Mismatch (SRM)

The design intended a **1/3 : 1/3 : 1/3** split. If the observed group sizes deviate from
that by more than chance allows, the assignment mechanism itself is broken — and nothing
downstream can be trusted. We test with a chi-square goodness-of-fit and use the industry
alarm threshold of **p < 0.001** (Microsoft reports ~6% of its experiments trip this check).

In [ ]:
counts = df["segment"].value_counts().sort_index()
print(counts.to_string(), "\n")

srm = srm_check(counts.values)  # equal split expected by default
print(f"chi-square = {srm['chi2']:.3f}, p = {srm['p_value']:.3f}")
print("SRM check:", "PASS - split is consistent with 1/3 each"
      if srm["pass"] else "FAIL - assignment mechanism suspect")

## 3. Covariate balance — the randomization audit

Every column measured **before** the email send (recency, spend history, gender of past
purchases, channel...) was fixed before the coin flip. So if randomization worked, the
three arms must be statistically indistinguishable on all of them.

This matters because balance is what lets us claim **causality**: if the arms are identical
in every respect except the email, any outcome difference can only be caused by the email.

In [ ]:
PRE_TREATMENT = ["recency", "history", "mens", "womens",
                 "newbie", "channel", "zip_code"]

# ANOVA across the 3 arms for numeric covariates, chi-square for categoricals.
# Strict alpha=0.001, same logic as SRM: we run many checks, so a 0.05
# threshold would flag 1-in-20 by pure chance.
balance = balance_check(df, "segment", PRE_TREATMENT)
balance.style.format({"statistic": "{:.3f}", "p_value": "{:.3f}"})

In [ ]:
# Visual version of the same audit: if randomization worked, the distribution of
# past spend should be three copies of the same curve. (Log scale because spend
# history is heavily right-skewed - a handful of customers spent thousands.)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for seg in df["segment"].unique():
    sns.kdeplot(np.log1p(df.loc[df["segment"] == seg, "history"]),
                label=seg, ax=axes[0])
axes[0].set_xlabel("log(1 + past-year spend $)")
axes[0].set_title("Past spend distribution by arm\n(should overlap perfectly)")
axes[0].legend()

# Same idea for a categorical covariate: channel mix per arm.
channel_mix = (df.groupby("segment")["channel"]
                 .value_counts(normalize=True).unstack())
channel_mix.plot(kind="bar", ax=axes[1], rot=0)
axes[1].set_title("Purchase channel mix by arm\n(bars should match across arms)")
axes[1].set_ylabel("share of arm")

fig.tight_layout()
fig.savefig("../assets/balance_checks.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. First (descriptive) look at outcomes by arm

We allow ourselves ONE descriptive table before the formal tests. No p-values here — that
is notebook 03's job, where multiple-comparison corrections are applied properly.

In [ ]:
outcome_table = (df.groupby("segment")[["visit", "conversion", "spend"]]
                   .mean()
                   .rename(columns={"visit": "visit rate",
                                    "conversion": "conversion rate",
                                    "spend": "mean spend $"}))
outcome_table.style.format({"visit rate": "{:.2%}",
                            "conversion rate": "{:.3%}",
                            "mean spend $": "${:.3f}"})

In [ ]:
# The chart version, with binomial standard-error bars so the eye gets a sense
# of uncertainty before the formal tests.
fig, ax = plt.subplots(figsize=(7.5, 4.5))

stats_ = df.groupby("segment")["conversion"].agg(["mean", "count"])
stats_["se"] = np.sqrt(stats_["mean"] * (1 - stats_["mean"]) / stats_["count"])
order = ["No E-Mail", "Womens E-Mail", "Mens E-Mail"]
stats_ = stats_.loc[order]

colors = ["#9e9e9e", "#e07a9b", "#4878cf"]  # control in grey, treatments colored
ax.bar(stats_.index, stats_["mean"] * 100,
       yerr=stats_["se"] * 100 * 1.96, capsize=6, color=colors)
ax.set_ylabel("2-week conversion rate (%)")
ax.set_title("Conversion by arm (error bars = 95% CI)")
for i, (idx, row) in enumerate(stats_.iterrows()):
    ax.text(i, row["mean"] * 100 + 0.02, f"{row['mean']:.2%}",
            ha="center", fontweight="bold")

fig.tight_layout()
fig.savefig("../assets/conversion_by_arm.png", dpi=150, bbox_inches="tight")
plt.show()

## Validation verdict

| Check | Result |
|---|---|
| Data integrity (missingness, ranges, funnel logic) | PASS |
| Sample Ratio Mismatch (chi-square vs 1/3:1/3:1/3) | PASS (p ≈ 0.90) |
| Covariate balance across 7 pre-treatment variables | PASS (all p > 0.4) |

The experiment is **valid**: the three arms are random samples of the same population, so
outcome differences can be attributed to the emails. The descriptive numbers *suggest* the
Men's email roughly doubles conversion — but that claim needs proper hypothesis tests with
multiple-comparison control, which is exactly what notebook `03_analysis.ipynb` does.